<a href="https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛣️ Smart City Road-Defect & Pothole Detection — YOLOv8 Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This notebook trains a custom **YOLOv8** model for real-time onboard edge detection of road potholes and defects on transit buses.

### Step 1: Check GPU & Install Dependencies

In [1]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib

Sat Sep 12 11:18:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Download Public Pothole Dataset

In [2]:
# Download benchmark road pothole dataset (Roboflow Universe / Public)
!curl -L "https://universe.roboflow.com/ds/t6yR1o0mR0?key=3sL2Zl6yGz" > roboflow.zip; unzip -q roboflow.zip -d ./pothole_dataset; rm roboflow.zip
!curl -L "https://universe.roboflow.com/ds/t6yR1o0mR0?key=3sL2Zl6yGz" > roboflow.zip
!unzip -q -o roboflow.zip -d ./pothole_dataset
!rm roboflow.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5489  100  5489    0     0  40859      0 --:--:-- --:--:-- --:--:-- 40962
[roboflow.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of roboflow.zip or
        roboflow.zip.zip, and cannot find roboflow.zip.ZIP, period.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5489  100  5489    0     0   107k      0 --:--:-- --:--:-- --:--:--  107k
[roboflow.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one d

### Step 3: Train YOLOv8 Model on GPU

In [3]:
import os
import glob
from ultralytics import YOLO

# 1. Clean up & clone verified YOLO pothole repository
!rm -rf /content/pothole_data
!git clone https://github.com/muhammetdinc/pothole-detection.git /content/pothole_data

# 2. Inspect folder structure and auto-generate the exact data.yaml
base_dir = "/content/pothole_data"

# Check available image folders
train_path = glob.glob(f"{base_dir}/**/train", recursive=True)
val_path = glob.glob(f"{base_dir}/**/val*", recursive=True)

train_dir = train_path[0] if train_path else f"{base_dir}/train"
val_dir = val_path[0] if val_path else f"{base_dir}/val"

# Write proper data.yaml with exact paths
yaml_path = "/content/pothole_data/dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(f"""
path: /content/pothole_data
train: {train_dir}
val: {val_dir}

names:
  0: pothole
""")

print("✅ Using verified config:\n" + open(yaml_path).read())

# 3. Train YOLOv8 on Tesla T4 GPU
model = YOLO('yolov8n.pt')
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='pothole_yolov8_model'
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Cloning into '/content/pothole_data'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/pothole_data/dataset.yaml'

### Step 4: Evaluate Model Performance (mAP, Loss & Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Validate the model
metrics = model.val()
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

# Display training loss curves & confusion matrix
results_img = cv2.imread('runs/detect/pothole_yolov8_model/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Loss & Validation Metrics')
    plt.show()

### Step 5: Test Model on Test Images

In [ ]:
# Run prediction on test set
preds = model.predict(source='./pothole_dataset/test/images', conf=0.5, save=True)
print("Predictions saved to runs/detect/predict/")

### Step 6: Download the Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/pothole_yolov8_model/weights/best.pt')